# Human-in-the-Loop (HITL) Pattern
### When certain decisions require human judgment

Not every decision can be automated. Some carry too much risk, require regulatory sign-off, or involve judgment that the business isn’t ready to delegate to an LLM. The HITL pattern lets agents handle the routine work and **pause at decision points** where a human must weigh in.

The agent doesn’t stop working — it prepares everything the human needs to make a fast, informed decision, then waits for the verdict before continuing.

### Business Use Case: AnyComp Telecom Enterprise Contract Approval

When a sales rep negotiates an enterprise deal, the agent reviews the terms against pricing policy. Deals with discounts under 15% are auto-approved. Deals with larger discounts pause for manager approval. The manager sees the full deal summary, the policy check, and the risk assessment — then decides.

### Two Scenarios in This Notebook

| Scenario | Discount | What Happens |
|---|---|---|
| **Happy path** | 10% (under threshold) | Agent auto-approves, generates contract |
| **HITL path** | 25% (over threshold) | Agent pauses, you decide, agent continues |

### Architecture

```
Deal Request → Agent prepares deal summary
                    │
              Check pricing policy
                    │
              Discount > threshold?
                    │
              ┌────┼────┐
              No          Yes
              │            │
         Auto-approve   ⏸ PAUSE — wait for human
              │            │
              │       Manager reviews
              │       [approve / reject / modify]
              │            │
              └────┼────┘
                    │
              Agent continues → generate contract / notify sales
```



## Install Dependencies

Run this cell first, then **restart the kernel** and continue.

In [ ]:
!pip install -q strands-agents>=1.34.0 strands-agents-tools boto3 "typing_extensions>=4.12.0"

## Install Dependencies

**Restart the kernel** after installing, then continue.

## Setup: Deal Data and Pricing Policy

In [ ]:
import json

REGION = 'us-east-1'
AUTO_APPROVE_THRESHOLD = 15  # percent discount

# Deal that will auto-approve (10% discount)
DEAL_AUTO = {
    "deal_id": "DEAL-2026-101",
    "customer": "Metro City Hospital",
    "product": "Enterprise 5G Connectivity Package",
    "annual_value": 180000,
    "discount_pct": 10,
    "term_years": 3,
    "sales_rep": "Sarah Kim",
}

# Deal that needs human approval (25% discount)
DEAL_HITL = {
    "deal_id": "DEAL-2026-102",
    "customer": "StateWide Logistics Corp",
    "product": "Enterprise IoT Fleet Management",
    "annual_value": 320000,
    "discount_pct": 25,
    "term_years": 5,
    "sales_rep": "James Wilson",
}

PRICING_POLICY = (
    "AnyComp Telecom Enterprise Pricing Policy:\n"
    "- Discounts up to 15%: auto-approved for deals over $100K/year\n"
    "- Discounts 15-30%: require Sales Director approval\n"
    "- Discounts over 30%: require VP Sales approval\n"
    "- Multi-year deals (3+ years): eligible for additional 5% loyalty bonus\n"
    "- All enterprise deals must include SLA terms and support tier"
)

print(f'Auto-approve threshold: {AUTO_APPROVE_THRESHOLD}% discount')
print(f'Deal 1: {DEAL_AUTO["deal_id"]} — {DEAL_AUTO["discount_pct"]}% discount (auto)')
print(f'Deal 2: {DEAL_HITL["deal_id"]} — {DEAL_HITL["discount_pct"]}% discount (HITL)')

## Implementation 1: Strands Agents

The agent has two tools: `review_deal` (checks against policy) and `request_human_approval` (pauses for human input via `input()`). The agent decides which path based on the discount threshold.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=REGION,
    max_tokens=4096,
)

@tool
def review_deal(deal_json: str) -> str:
    """Review a deal against pricing policy and determine if it needs human approval.
    Args:
        deal_json: JSON string with deal details
    """
    deal = json.loads(deal_json)
    discount = deal.get('discount_pct', 0)
    annual = deal.get('annual_value', 0)
    net_annual = annual * (1 - discount / 100)
    total_contract = net_annual * deal.get('term_years', 1)

    review = {
        "deal_id": deal["deal_id"],
        "customer": deal["customer"],
        "list_price": f"${annual:,.0f}/year",
        "discount": f"{discount}%",
        "net_price": f"${net_annual:,.0f}/year",
        "total_contract_value": f"${total_contract:,.0f}",
        "needs_approval": discount > AUTO_APPROVE_THRESHOLD,
        "approval_level": "Auto" if discount <= 15 else "Sales Director" if discount <= 30 else "VP Sales",
    }
    return json.dumps(review, indent=2)

@tool
def request_human_approval(deal_summary: str) -> str:
    """Pause and request human approval for a deal that exceeds the auto-approve threshold.
    Args:
        deal_summary: Summary of the deal for the approver to review
    """
    print("\n" + "═" * 60)
    print("🛑 HUMAN APPROVAL REQUIRED")
    print("═" * 60)
    print(deal_summary)
    print("═" * 60)

    decision = input("\n🏛️ Decision (approve/reject/modify): ").strip().lower()
    notes = input("📝 Notes: ").strip()

    return json.dumps({
        "decision": decision,
        "notes": notes,
        "approved_by": "Manager (notebook user)",
    })

@tool
def generate_contract_confirmation(deal_id: str, decision: str, notes: str) -> str:
    """Generate a contract confirmation or rejection notice.
    Args:
        deal_id: The deal identifier
        decision: approve, reject, or modify
        notes: Approver notes
    """
    if decision == "approve":
        return f"CONTRACT CONFIRMED: {deal_id}. Generating contract documents. Sales rep notified."
    elif decision == "reject":
        return f"DEAL REJECTED: {deal_id}. Reason: {notes}. Sales rep notified to renegotiate."
    else:
        return f"DEAL MODIFICATION REQUESTED: {deal_id}. Changes: {notes}. Returning to sales rep."


deal_agent = Agent(
    model=model,
    system_prompt=(
        "You are a deal review agent for AnyComp Telecom. "
        "For each deal:\n"
        "1. Call review_deal to check it against pricing policy\n"
        "2. If needs_approval is true, call request_human_approval with the deal summary\n"
        "3. If needs_approval is false, auto-approve\n"
        "4. Call generate_contract_confirmation with the final decision\n"
        "5. Summarize the outcome"
    ),
    tools=[review_deal, request_human_approval, generate_contract_confirmation],
)

### Scenario 1: Auto-Approve (10% discount)

This deal is under the threshold — the agent reviews and auto-approves without pausing.

In [ ]:
print("=" * 60)
print("SCENARIO 1: Auto-Approve Path")
print("=" * 60)
print(deal_agent(f'Process this deal:\n{json.dumps(DEAL_AUTO, indent=2)}\n\nPricing policy:\n{PRICING_POLICY}'))

### Scenario 2: HITL Path (25% discount)

This deal exceeds the threshold. The agent will pause and ask **you** to approve or reject. Type your decision when prompted.

In [ ]:
print("=" * 60)
print("SCENARIO 2: Human-in-the-Loop Path")
print("=" * 60)
print(deal_agent(f'Process this deal:\n{json.dumps(DEAL_HITL, indent=2)}\n\nPricing policy:\n{PRICING_POLICY}'))

## Production Considerations

In this notebook, the human decision happens via `input()` in the cell. In production:

| This Notebook | Production |
|---|---|
| `input()` prompt | Web UI, email link, Slack bot, or Amazon Quick Automate task queue |
| Synchronous (cell blocks) | Asynchronous (Step Functions callback, webhook) |
| Single reviewer | Role-based routing to the right approver |
| Immediate | Hours or days — the workflow waits without consuming compute |

The pattern is the same regardless of the human interface: **agent prepares → pauses at decision point → human decides → agent continues**.